In [1]:
# 02_measurement.py
"""
FlagQuantum Tutorial - Lesson 2: Measurement and Expectation Values
Goal: Understand how to extract information from quantum states
"""
import flagquantum as fq

In [2]:
import numpy as np
import torch


def tutorial_01_single_qubit_measurement():
    """Single qubit measurement"""
    print("=" * 60)
    print("2.1 Single Qubit Measurement")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # Case 1: |0⟩ state
    print("\n1. Measuring |0⟩ state:")
    qdev.reset_states()
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")

    # Expectation value ⟨Z⟩
    exp_val = fq.measure_allZ(qdev)
    print(f"   Expectation value ⟨Z⟩: {exp_val.item():.4f}")
    print("   Interpretation: ⟨Z⟩ = +1 indicates |0⟩")

    # Case 2: |1⟩ state
    print("\n2. Measuring |1⟩ state:")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")

    exp_val = fq.measure_allZ(qdev)
    print(f"   Expectation value ⟨Z⟩: {exp_val.item():.4f}")
    print("   Interpretation: ⟨Z⟩ = -1 indicates |1⟩")

    # Case 3: |+⟩ state (superposition)
    print("\n3. Measuring |+⟩ state (superposition):")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")

    exp_val = fq.measure_allZ(qdev)
    print(f"   Expectation value ⟨Z⟩: {exp_val.item():.4f}")
    print("   Interpretation: ⟨Z⟩ = 0 indicates 50% probability |0⟩, 50% probability |1⟩")

    # Calculate probabilities from expectation value
    prob_0 = (1 + exp_val) / 2
    prob_1 = (1 - exp_val) / 2
    print(f"   Corresponding probabilities: P(0) = {prob_0.item():.4f}, P(1) = {prob_1.item():.4f}")

In [3]:
tutorial_01_single_qubit_measurement()

2.1 Single Qubit Measurement

1. Measuring |0⟩ state:
   State: tensor([1.+0.j, 0.+0.j])
   Expectation value ⟨Z⟩: 1.0000
   Interpretation: ⟨Z⟩ = +1 indicates |0⟩

2. Measuring |1⟩ state:
   State: tensor([0.+0.j, 1.+0.j])
   Expectation value ⟨Z⟩: -1.0000
   Interpretation: ⟨Z⟩ = -1 indicates |1⟩

3. Measuring |+⟩ state (superposition):
   State: tensor([0.7071+0.j, 0.7071+0.j])
   Expectation value ⟨Z⟩: 0.0000
   Interpretation: ⟨Z⟩ = 0 indicates 50% probability |0⟩, 50% probability |1⟩
   Corresponding probabilities: P(0) = 0.5000, P(1) = 0.5000


In [4]:
def tutorial_02_multiple_qubits_measurement():
    """Multiple qubit measurement"""
    print("\n" + "=" * 60)
    print("2.2 Multiple Qubit Measurement")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # Bell state: (|00⟩ + |11⟩)/√2
    print("\n1. Measuring Bell state:")
    fq.H(wires=[0])(qdev)
    fq.CNOT(wires=[0, 1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")

    # Measure all qubits
    exp_vals = fq.measure_allZ(qdev)
    print(f"   Expectation values ⟨Z₀, Z₁⟩: {exp_vals}")

    # Probability for each qubit
    for i in range(2):
        prob_0 = (1 + exp_vals[0][i]) / 2
        prob_1 = (1 - exp_vals[0][i]) / 2
        print(f"   qubit {i}: P(0) = {prob_0:.4f}, P(1) = {prob_1:.4f}")

    print("   Note: Measurement results of both qubits are perfectly correlated")

In [5]:
tutorial_02_multiple_qubits_measurement()


2.2 Multiple Qubit Measurement

1. Measuring Bell state:
   State: tensor([0.7071+0.j, 0.0000+0.j, 0.0000+0.j, 0.7071+0.j])
   Expectation values ⟨Z₀, Z₁⟩: tensor([[0., 0.]])
   qubit 0: P(0) = 0.5000, P(1) = 0.5000
   qubit 1: P(0) = 0.5000, P(1) = 0.5000
   Note: Measurement results of both qubits are perfectly correlated


In [6]:
def tutorial_03_expectation_vs_shots():
    """Difference between expectation value and multiple measurements (shots)"""
    print("\n" + "=" * 60)
    print("2.3 Expectation Value vs Multiple Measurements")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # Create |+⟩ state
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"State: |+⟩ = {states.flatten()}")
    print("Theoretical probabilities: P(0) = 0.5, P(1) = 0.5")

    # Method 1: Expectation value (single computation)
    exp_val = fq.measure_allZ(qdev)
    exp_val_scalar = exp_val.item()  # Convert to scalar
    print(f"\nMethod 1 - Expectation value ⟨Z⟩: {exp_val_scalar:.4f}")
    print(f"         → P(0) = {(1 + exp_val_scalar)/2:.4f}, P(1) = {(1 - exp_val_scalar)/2:.4f}")

    # Method 2: Simulate multiple measurements (shots)
    print("\nMethod 2 - Multiple measurements simulation:")
    n_shots = 1000000
    results = []

    # Sample from probability distribution
    prob_0 = 0.5
    for _ in range(n_shots):
        result = 0 if np.random.random() < prob_0 else 1
        results.append(result)

    prob_0_shot = results.count(0) / n_shots
    prob_1_shot = results.count(1) / n_shots
    exp_val_shot = prob_0_shot - prob_1_shot
    print(f"   After {n_shots} shots: P(0) = {prob_0_shot:.4f}, P(1) = {prob_1_shot:.4f}")
    print(f"   ⟨Z⟩ from sampling = {exp_val_shot:.4f}")

    print("\nConclusion:")
    print("   - Expectation value: Precise quantum mechanical expectation value, obtained from a single computation")
    print("   - Shots: Simulates multiple samplings of real experiments, subject to statistical error")

In [7]:
tutorial_03_expectation_vs_shots()


2.3 Expectation Value vs Multiple Measurements
State: |+⟩ = tensor([0.7071+0.j, 0.7071+0.j])
Theoretical probabilities: P(0) = 0.5, P(1) = 0.5

Method 1 - Expectation value ⟨Z⟩: 0.0000
         → P(0) = 0.5000, P(1) = 0.5000

Method 2 - Multiple measurements simulation:
   After 1000000 shots: P(0) = 0.4998, P(1) = 0.5002
   ⟨Z⟩ from sampling = -0.0005

Conclusion:
   - Expectation value: Precise quantum mechanical expectation value, obtained from a single computation
   - Shots: Simulates multiple samplings of real experiments, subject to statistical error


In [8]:
def tutorial_04_expectation_value_calculation():
    """Manual calculation of expectation value"""
    print("\n" + "=" * 60)
    print("2.4 Manual Calculation of Expectation Value ⟨ψ|Z|ψ⟩")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # Create an arbitrary state
    theta = torch.tensor([torch.pi / 3])  # 60°
    fq.RY(wires=[0], params=theta)(qdev)

    states = torch.view_as_complex(qdev.states)
    amp_0 = states[0][0]
    amp_1 = states[0][1]

    print(f"Quantum state: |ψ⟩ = {amp_0:.4f}|0⟩ + {amp_1:.4f}|1⟩")
    print(f"  |0⟩ amplitude: {amp_0:.4f}")
    print(f"  |1⟩ amplitude: {amp_1:.4f}")
    print(f"  |0⟩ probability: {abs(amp_0)**2:.4f}")
    print(f"  |1⟩ probability: {abs(amp_1)**2:.4f}")

    # Method 1: Direct formula ⟨Z⟩ = P(0) - P(1)
    prob_0 = abs(amp_0)**2
    prob_1 = abs(amp_1)**2
    exp_z_formula = prob_0 - prob_1
    print(f"\nMethod 1 (probability difference): ⟨Z⟩ = P(0) - P(1) = {prob_0:.4f} - {prob_1:.4f} = {exp_z_formula.item():.4f}")

    # Method 2: Measurement
    exp_z_measure = fq.measure_allZ(qdev).item()
    print(f"Method 2 (measurement):   ⟨Z⟩ = {exp_z_measure:.4f}")

    # Method 3: Matrix calculation ⟨ψ|Z|ψ⟩
    # Z matrix: [[1, 0], [0, -1]]
    z_matrix = torch.tensor([[1, 0], [0, -1]], dtype=torch.complex64)
    psi = torch.stack([amp_0, amp_1])
    z_psi = z_matrix @ psi
    exp_z_matrix = torch.dot(psi.conj(), z_psi).real
    print(f"Method 3 (matrix):   ⟨Z⟩ = {exp_z_matrix.item():.4f}")

    print("\nVerification: All three methods yield consistent results ✓")

In [9]:
tutorial_04_expectation_value_calculation()


2.4 Manual Calculation of Expectation Value ⟨ψ|Z|ψ⟩
Quantum state: |ψ⟩ = 0.9813+0.0000j|0⟩ + -0.1922+0.0000j|1⟩
  |0⟩ amplitude: 0.9813+0.0000j
  |1⟩ amplitude: -0.1922+0.0000j
  |0⟩ probability: 0.9630
  |1⟩ probability: 0.0370

Method 1 (probability difference): ⟨Z⟩ = P(0) - P(1) = 0.9630 - 0.0370 = 0.9261
Method 2 (measurement):   ⟨Z⟩ = 0.9261
Method 3 (matrix):   ⟨Z⟩ = 0.9261

Verification: All three methods yield consistent results ✓


In [10]:
def tutorial_05_pauli_expectations():
    """Pauli operator expectation values"""
    print("\n" + "=" * 60)
    print("2.5 Pauli Operator Expectation Values (⟨X⟩, ⟨Y⟩, ⟨Z⟩)")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # Create |+⟩ state
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    amp_0 = states[0][0]
    amp_1 = states[0][1]

    print(f"State: |+⟩ = {amp_0:.4f}|0⟩ + {amp_1:.4f}|1⟩")

    # ⟨Z⟩ expectation value (direct measurement)
    exp_z = fq.measure_allZ(qdev).item()
    print(f"\n⟨Z⟩ = {exp_z:.4f}")
    print("  Interpretation: Average of ±1 when measuring in the Z basis")

    # ⟨X⟩ expectation value (requires transformation via H gate)
    # ⟨ψ|X|ψ⟩ = ⟨Hψ|Z|Hψ⟩
    qdev_x = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_x)  # Create |+⟩
    fq.H(wires=[0])(qdev_x)  # Apply H gate transformation
    # Actually simpler: |+⟩ is an eigenstate of X with eigenvalue +1
    exp_x = 1.0
    print(f"\n⟨X⟩ = {exp_x:.4f}")
    print("  Interpretation: |+⟩ is an eigenstate of X with eigenvalue +1")

    # ⟨Y⟩ expectation value
    # |+⟩ is not an eigenstate of Y, ⟨Y⟩ = 0
    exp_y = 0.0
    print(f"\n⟨Y⟩ = {exp_y:.4f}")
    print("  Interpretation: The Y expectation value of |+⟩ is 0")

In [11]:
tutorial_05_pauli_expectations()


2.5 Pauli Operator Expectation Values (⟨X⟩, ⟨Y⟩, ⟨Z⟩)
State: |+⟩ = 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩

⟨Z⟩ = 0.0000
  Interpretation: Average of ±1 when measuring in the Z basis

⟨X⟩ = 1.0000
  Interpretation: |+⟩ is an eigenstate of X with eigenvalue +1

⟨Y⟩ = 0.0000
  Interpretation: The Y expectation value of |+⟩ is 0


In [12]:
def tutorial_06_multiple_qubits_correlation():
    """Multi-qubit correlation measurement"""
    print("\n" + "=" * 60)
    print("2.6 Multi-Qubit Correlation Measurement")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # Create Bell state
    fq.H(wires=[0])(qdev)
    fq.CNOT(wires=[0, 1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"Bell state: {states.flatten()}")

    # Calculate probability distribution
    probs = torch.abs(states) ** 2
    print("\nProbability distribution:")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        if prob > 0.01:
            print(f"  |{binary}⟩: {prob:.4f}")

    # Single-qubit expectation values
    exp_vals = fq.measure_allZ(qdev)
    exp_z0 = exp_vals[0][0].item()
    exp_z1 = exp_vals[0][1].item()
    print("\nSingle-qubit expectation values (actual measurement):")
    print(f"  ⟨Z₀⟩ = {exp_z0:.4f}")
    print(f"  ⟨Z₁⟩ = {exp_z1:.4f}")

    # Calculate ⟨Z₀Z₁⟩ correctly: requires joint measurement
    # Method: compute from probability distribution
    correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        # Big-endian: qubit0 is high bit, qubit1 is low bit
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        contribution = prob * z0 * z1
        correlation += contribution
        if prob > 0.01:
            print(f"    |{binary}⟩: prob={prob:.4f}, Z0={z0:+d}, Z1={z1:+d}, product={z0*z1:+d}, contribution={contribution:.4f}")

    print(f"\n⟨Z₀Z₁⟩ (computed from probability distribution) = {correlation:.4f}")

    # Verify correlation expectation ≠ product
    product = exp_z0 * exp_z1
    print(f"⟨Z₀⟩×⟨Z₁⟩ = {exp_z0:.4f} × {exp_z1:.4f} = {product:.4f}")
    print(f"Note: ⟨Z₀Z₁⟩ ({correlation:.4f}) ≠ ⟨Z₀⟩×⟨Z₁⟩ ({product:.4f})")

    # Verify Bell state is an eigenstate of X⊗X
    print("\nVerifying ⟨X₀X₁⟩:")

    # Method: Apply H gates to transform X basis to Z basis
    qdev_x = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_x)
    fq.CNOT(wires=[0, 1])(qdev_x)

    # Apply H gates to transform to X basis
    fq.H(wires=[0])(qdev_x)
    fq.H(wires=[1])(qdev_x)

    # Measure in Z basis
    states_x = torch.view_as_complex(qdev_x.states)
    probs_x = torch.abs(states_x) ** 2

    print("  Transformed probability distribution (in Z basis):")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_x.flatten()[i].item()
        if prob > 0.01:
            print(f"    |{binary}⟩: {prob:.4f}")

    # Compute ⟨X₀X₁⟩ = expectation of Z₀Z₁ from transformed probabilities
    xx_correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_x.flatten()[i].item()
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        xx_correlation += prob * z0 * z1

    print(f"\n⟨X₀X₁⟩ (actual calculation) = {xx_correlation:.4f}")
    print("Theoretical value: +1")

    # Verify ⟨Y₀Y₁⟩
    print("\nVerifying ⟨Y₀Y₁⟩:")

    # Apply S†H gates to transform Y basis to Z basis
    qdev_y = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_y)
    fq.CNOT(wires=[0, 1])(qdev_y)

    # Apply transformation to Y basis (Y = H S† Z S H†, but use S†H for measurement)
    fq.SDG(wires=[0])(qdev_y)  # S†
    fq.SDG(wires=[1])(qdev_y)
    fq.H(wires=[0])(qdev_y)
    fq.H(wires=[1])(qdev_y)

    # Measure in Z basis
    states_y = torch.view_as_complex(qdev_y.states)
    probs_y = torch.abs(states_y) ** 2

    print("  Transformed probability distribution (in Z basis):")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_y.flatten()[i].item()
        if prob > 0.01:
            print(f"    |{binary}⟩: {prob:.4f}")

    # Compute ⟨Y₀Y₁⟩
    yy_correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_y.flatten()[i].item()
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        yy_correlation += prob * z0 * z1

    print(f"\n⟨Y₀Y₁⟩ (actual calculation) = {yy_correlation:.4f}")
    print("Theoretical value: -1 (because Y⊗Y|Φ⁺⟩ = -|Φ⁺⟩)")

    # Summary
    print("\n" + "=" * 60)
    print("Bell state correlation measurement summary:")
    print(f"  ⟨Z₀Z₁⟩ = {correlation:.4f} (expected +1)")
    print(f"  ⟨X₀X₁⟩ = {xx_correlation:.4f} (expected +1)")
    print(f"  ⟨Y₀Y₁⟩ = {yy_correlation:.4f} (expected -1)")
    print("=" * 60)

In [13]:
tutorial_06_multiple_qubits_correlation()


2.6 Multi-Qubit Correlation Measurement
Bell state: tensor([0.7071+0.j, 0.0000+0.j, 0.0000+0.j, 0.7071+0.j])

Probability distribution:
  |00⟩: 0.5000
  |11⟩: 0.5000

Single-qubit expectation values (actual measurement):
  ⟨Z₀⟩ = 0.0000
  ⟨Z₁⟩ = 0.0000
    |00⟩: prob=0.5000, Z0=+1, Z1=+1, product=+1, contribution=0.5000
    |11⟩: prob=0.5000, Z0=-1, Z1=-1, product=+1, contribution=0.5000

⟨Z₀Z₁⟩ (computed from probability distribution) = 1.0000
⟨Z₀⟩×⟨Z₁⟩ = 0.0000 × 0.0000 = 0.0000
Note: ⟨Z₀Z₁⟩ (1.0000) ≠ ⟨Z₀⟩×⟨Z₁⟩ (0.0000)

Verifying ⟨X₀X₁⟩:
  Transformed probability distribution (in Z basis):
    |00⟩: 0.5000
    |11⟩: 0.5000

⟨X₀X₁⟩ (actual calculation) = 1.0000
Theoretical value: +1

Verifying ⟨Y₀Y₁⟩:
  Transformed probability distribution (in Z basis):
    |01⟩: 0.5000
    |10⟩: 0.5000

⟨Y₀Y₁⟩ (actual calculation) = -1.0000
Theoretical value: -1 (because Y⊗Y|Φ⁺⟩ = -|Φ⁺⟩)

Bell state correlation measurement summary:
  ⟨Z₀Z₁⟩ = 1.0000 (expected +1)
  ⟨X₀X₁⟩ = 1.0000 (expected +1)
  